# Patient-Level Train, Validation, and Test Splitting

This notebook validates patient identifiers before creating leakage-safe train, validation, and test datasets. Only rows marked `usable == True` are included. The original manifest and raw images are read only and are not modified.

A patient-level binary target is used only to stratify patient groups: a patient is treated as AMD-positive if any usable image for that patient has `target_amd == 1`. The original row-level target remains unchanged in every split.

## 1. Load the cleaned manifest

The split is built from `data/processed/manifest.csv` and is written to `data/processed/train.csv`, `validation.csv`, and `test.csv`.

In [1]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

RANDOM_SEED = 42
TRAIN_FRACTION = 0.70
VALIDATION_FRACTION = 0.15
TEST_FRACTION = 0.15

def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "data" / "raw" / "OCTDL").is_dir():
            return candidate
    raise FileNotFoundError("Could not find a project root containing data/raw/OCTDL.")

PROJECT_ROOT = find_project_root()
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed"
MANIFEST_PATH = PROCESSED_ROOT / "manifest.csv"

if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(f"Manifest not found: {MANIFEST_PATH}")

manifest = pd.read_csv(MANIFEST_PATH)
required_columns = {"patient_id", "target_amd", "usable", "disease"}
missing_columns = required_columns.difference(manifest.columns)
if missing_columns:
    raise ValueError(f"Manifest is missing columns: {sorted(missing_columns)}")

usable = manifest.loc[manifest["usable"].astype(bool)].copy()
usable["target_amd"] = pd.to_numeric(usable["target_amd"], errors="raise").astype(int)
print(f"Manifest rows: {len(manifest):,}")
print(f"Usable rows selected for splitting: {len(usable):,}")

Manifest rows: 2,064
Usable rows selected for splitting: 2,062


## 2. Validate patient ID reliability before splitting

This gate runs before any split is assigned. IDs are considered invalid when they are null, empty, common placeholder values, non-numeric/non-string values, or non-positive numeric values.

In [2]:
patient_text = usable["patient_id"].astype("string").str.strip()
placeholder_values = {"", "0", "0.0", "nan", "none", "null", "unknown", "unk", "na", "n/a"}
placeholder_mask = patient_text.str.lower().isin(placeholder_values)
numeric_ids = pd.to_numeric(patient_text, errors="coerce")
invalid_mask = patient_text.isna() | placeholder_mask | (numeric_ids.notna() & (numeric_ids <= 0))

total_unique_patients = usable["patient_id"].nunique(dropna=True)
images_per_patient = usable.groupby("patient_id", dropna=False).size()
repeated_patients = int((images_per_patient > 1).sum())
patient_label_counts = usable.groupby("patient_id")["target_amd"].nunique()
label_conflict_patients = patient_label_counts[patient_label_counts > 1]

print("PATIENT ID VALIDATION RESULTS")
print("-" * 32)
print(f"Total unique patient IDs: {total_unique_patients:,}")
print(f"Images with missing or invalid patient IDs: {int(invalid_mask.sum()):,}")
print(f"Placeholder patient-ID values: {int(placeholder_mask.sum()):,}")
print(f"Patients with multiple images: {repeated_patients:,}")
print(f"Maximum images for one patient: {int(images_per_patient.max()):,}")
print(f"Patients with mixed AMD/non-AMD row labels: {len(label_conflict_patients):,}")
print("\nImages per patient distribution:")
print(images_per_patient.value_counts().sort_index().to_string())

patient_ids_reliable = bool(len(usable) > 0 and invalid_mask.sum() == 0)
print(f"\nPatient IDs reliable enough for patient-level splitting: {patient_ids_reliable}")
if not patient_ids_reliable:
    raise ValueError("Patient IDs are unreliable. Do not split individual images; obtain corrected IDs or use a documented alternative grouping key.")
if len(label_conflict_patients):
    print("Note: mixed patient labels exist. Stratification will use any-AMD at patient level, while row labels remain unchanged.")

PATIENT ID VALIDATION RESULTS
--------------------------------
Total unique patient IDs: 820
Images with missing or invalid patient IDs: 0
Placeholder patient-ID values: 0
Patients with multiple images: 568
Maximum images for one patient: 10
Patients with mixed AMD/non-AMD row labels: 1

Images per patient distribution:
1     252
2     280
3      93
4     103
5      36
6      32
7      10
8      10
9       3
10      1

Patient IDs reliable enough for patient-level splitting: True
Note: mixed patient labels exist. Stratification will use any-AMD at patient level, while row labels remain unchanged.


## 3. Build patient groups and stratification labels

Every patient is assigned once. The patient-level stratification target is `max(target_amd)`, which ensures that a patient with any AMD-labeled image is represented in the AMD patient stratum.

In [3]:
patient_summary = (
    usable.groupby("patient_id", as_index=False)
    .agg(
        patient_target=("target_amd", "max"),
        image_count=("target_amd", "size"),
    )
)

print("Patient-level stratification counts:")
print(patient_summary["patient_target"].value_counts().sort_index().rename({0: "non-AMD patients", 1: "AMD patients"}).to_string())
print(f"Total patient groups: {len(patient_summary):,}")

Patient-level stratification counts:
patient_target
non-AMD patients    399
AMD patients        421
Total patient groups: 820


## 4. Create deterministic patient-level splits

Patients are shuffled within each binary stratum using seed 42, then assigned approximately 70%/15%/15%. No image-level random split is used.

In [4]:
rng = np.random.default_rng(RANDOM_SEED)
patient_summary["split"] = pd.NA

for patient_target in [0, 1]:
    stratum = patient_summary.loc[patient_summary["patient_target"] == patient_target].copy()
    shuffled_positions = rng.permutation(len(stratum))
    shuffled_ids = stratum.iloc[shuffled_positions]["patient_id"].tolist()
    n_patients = len(shuffled_ids)
    train_count = int(round(n_patients * TRAIN_FRACTION))
    validation_count = int(round(n_patients * VALIDATION_FRACTION))
    train_ids = shuffled_ids[:train_count]
    validation_ids = shuffled_ids[train_count:train_count + validation_count]
    test_ids = shuffled_ids[train_count + validation_count:]
    patient_summary.loc[patient_summary["patient_id"].isin(train_ids), "split"] = "train"
    patient_summary.loc[patient_summary["patient_id"].isin(validation_ids), "split"] = "validation"
    patient_summary.loc[patient_summary["patient_id"].isin(test_ids), "split"] = "test"

if patient_summary["split"].isna().any():
    raise RuntimeError("At least one patient was not assigned to a split.")

split_lookup = patient_summary.set_index("patient_id")["split"]
usable["split"] = usable["patient_id"].map(split_lookup)
if usable["split"].isna().any():
    raise RuntimeError("At least one usable image was not assigned through its patient ID.")

split_paths = {
    "train": PROCESSED_ROOT / "train.csv",
    "validation": PROCESSED_ROOT / "validation.csv",
    "test": PROCESSED_ROOT / "test.csv",
}
for split_name, output_path in split_paths.items():
    usable.loc[usable["split"] == split_name].drop(columns="split").to_csv(output_path, index=False)
    print(f"Saved {split_name}: {output_path}")

Saved train: C:\Users\vibhu\OneDrive\图片\文档\AMD-Digital-Twin\data\processed\train.csv
Saved validation: C:\Users\vibhu\OneDrive\图片\文档\AMD-Digital-Twin\data\processed\validation.csv
Saved test: C:\Users\vibhu\OneDrive\图片\文档\AMD-Digital-Twin\data\processed\test.csv


## 5. Verify patient leakage and summarize split quality

The final checks compare patient-ID sets between every pair of splits. A passed leakage check requires all three intersections to be empty.

In [5]:
split_frames = {name: usable.loc[usable["split"] == name].copy() for name in split_paths}
patient_sets = {name: set(frame["patient_id"]) for name, frame in split_frames.items()}
overlap_counts = {
    "train_validation": len(patient_sets["train"] & patient_sets["validation"]),
    "train_test": len(patient_sets["train"] & patient_sets["test"]),
    "validation_test": len(patient_sets["validation"] & patient_sets["test"]),
}

def class_percentage(frame: pd.DataFrame, target: int) -> float:
    return 100.0 * float((frame["target_amd"] == target).sum()) / len(frame) if len(frame) else 0.0

report_rows = []
for split_name in ["train", "validation", "test"]:
    frame = split_frames[split_name]
    report_rows.append({
        "split": split_name,
        "images": len(frame),
        "unique_patients": frame["patient_id"].nunique(),
        "amd_images": int((frame["target_amd"] == 1).sum()),
        "non_amd_images": int((frame["target_amd"] == 0).sum()),
        "amd_percent": class_percentage(frame, 1),
        "non_amd_percent": class_percentage(frame, 0),
    })
split_report = pd.DataFrame(report_rows)
split_report.to_csv(PROCESSED_ROOT / "split_report.csv", index=False)
pd.DataFrame([overlap_counts]).to_csv(PROCESSED_ROOT / "split_patient_overlap.csv", index=False)

print("FINAL PATIENT-LEVEL SPLIT REPORT")
print("-" * 40)
display(split_report.style.format({"amd_percent": "{:.2f}%", "non_amd_percent": "{:.2f}%"}))
print("Overlapping patients between splits:")
for pair, count in overlap_counts.items():
    print(f"{pair}: {count}")

leakage_passed = all(count == 0 for count in overlap_counts.values())
print(f"\nPatient-level data leakage check passed: {leakage_passed}")
if not leakage_passed:
    raise AssertionError("Patient overlap detected between split files.")

FINAL PATIENT-LEVEL SPLIT REPORT
----------------------------------------


,split,images,unique_patients,amd_images,non_amd_images,amd_percent,non_amd_percent
0,train,1428,574,858,570,60.08%,39.92%
1,validation,327,123,197,130,60.24%,39.76%
2,test,307,123,174,133,56.68%,43.32%


Overlapping patients between splits:
train_validation: 0
train_test: 0
validation_test: 0

Patient-level data leakage check passed: True
